[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lslumass/HyresBuilder/blob/dev/HyRes_Quickstart.ipynb)

# HyRes Coarse-Grained Protein Simulation — Quick-Start (Google Colab)

This notebook lets you build a **HyRes** coarse-grained (CG) model of a disordered protein or peptide directly from its amino acid sequence, and run a short OpenMM molecular dynamics simulation on a free Colab GPU — no local installation required.

It uses:
- **[HyresBuilder](https://github.com/lslumass/HyresBuilder)** — builds HyRes/iConRNA structures and PSF topology files (ChenLab, UMass Amherst)
- **[OpenMM](https://openmm.org/)** — runs the molecular dynamics simulation

**Relevant papers:**
1. Liu & Chen, *Phys. Chem. Chem. Phys.*, 2017, 19, 32421 — original HyRes model
2. Zhang, Liu & Chen, *J. Chem. Inf. Model.*, 2022, 62, 4523 — HyRes-II
3. Zhang, Li, Gong & Chen, *J. Am. Chem. Soc.*, 2024, 146, 342 — HyRes-GPU
4. Li & Chen, *PNAS*, 2025, 122, e2504583122

**Before you start:** go to `Runtime → Change runtime type` and select a **GPU** (e.g. T4). This notebook is also configured to request a GPU runtime automatically when opened.

---
### Hot to run a demon here
1. Under the following "Varibles" section, give the name, sequence, and other conditions.
2. Click Run all.
3. This demo runs for 200 ns.


# Variables

In [ ]:
# IDP name
NAME = "demo"

# IDP sequence
SEQUENCE = "GSMASNNTASIAQARKLVEQLKMEANIDRIKVSKAAADLMAYCEAHAKEDPLLTPVPASENPFREKKFFCAIL"

# set up the terminal charge status
TERMINAL = "charged"

# simulation parameters
TEMP = 298   # temperature in Kelvin
SALT = 150   # concentration of NaCl in mM
OUT  = "demo_run"   # output name

## 0. Check the GPU runtime

In [ ]:
!nvidia-smi

## 1. Install the environment

In [ ]:
%%bash
set -e

# Remove any previous partial/old install (e.g. from an earlier attempt) so we
# always end up with a clean Miniforge (conda-forge-only) environment.
if [ -d /opt/conda ] && [ ! -f /opt/conda/.miniforge_ok ]; then
  rm -rf /opt/conda
fi

if [ ! -d /opt/conda ]; then
  echo "Downloading Miniforge..."
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
  bash /tmp/miniforge.sh -b -p /opt/conda
  touch /opt/conda/.miniforge_ok
fi
export PATH=/opt/conda/bin:$PATH

# Belt-and-suspenders: make sure the Anaconda "defaults" channels (which now
# require a ToS click-through) are never consulted, only conda-forge.
conda config --system --remove-key channels 2>/dev/null || true
conda config --system --add channels conda-forge
conda config --system --set channel_priority strict

if ! conda env list | grep -q "^hyres "; then
  conda create -n hyres -y python=3.9
fi
echo "Done."


In [ ]:
%%bash
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres

# psfgen is conda-forge only
conda install -y -c conda-forge psfgen numpy numba mdanalysis

# OpenMM with the CUDA platform (pip wheels bundle CUDA 12 support directly)
pip install -q "openmm[cuda12]"

# HyresBuilder itself, straight from GitHub
pip install -q "git+https://github.com/lslumass/HyresBuilder.git"

echo "----- OpenMM installation test -----"
python -m openmm.testInstallation


In [ ]:
%%bash
# Also grab a local clone of the repo — we need scripts/run_demo.py,
# which is not installed as part of the pip package.
if [ ! -d HyresBuilder_repo ]; then
  git clone -q https://github.com/lslumass/HyresBuilder.git HyresBuilder_repo
fi
ls HyresBuilder_repo/scripts


## 2. Prepare pdb and psf files

In [ ]:
import os
os.makedirs("run", exist_ok=True)
with open("run/build_structure.py", "w") as f:
    f.write(f'''
from HyresBuilder import PeptideBuilder
PeptideBuilder.build_peptide("{NAME}", "{SEQUENCE}")
print("Built {NAME}.pdb")
''')
print("Wrote run/build_structure.py for", NAME)


In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres
cd run
python build_structure.py
ls -la


In [ ]:


with open("run/build_psf.py", "w") as f:
    f.write(f'''
from HyresBuilder import GenPsf
GenPsf.genpsf("{NAME}.pdb", "{NAME}.psf", terminal="{TERMINAL}")
print("Built {NAME}.psf")
''')
print("Wrote run/build_psf.py")


In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres
cd run
python build_psf.py
ls -la


## 4. Run a short HyRes MD simulation

We use HyresBuilder's own `run_demo.py`, exactly as it's set up in the [HyresBuilder repo](https://github.com/lslumass/HyresBuilder) (`dev` branch), rather than reimplementing the force field ourselves — this guarantees the simulation matches the published, benchmarked HyRes setup.

The demo below runs a single chain in a **non-periodic** box (`-e non`), appropriate for a quick single-IDP test.

In [ ]:
cmd = (
    f"python -u HyresBuilder_repo/scripts/run_demo.py "
    f"-c run/{NAME}.pdb -p run/{NAME}.psf -o run/{OUT} "
    f"-t {TEMP} -e non -s {SALT}"
)
print(cmd)


In [ ]:
%%bash
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres
python HyresBuilder_repo/scripts/run_demo.py -c run/demo.pdb -p run/demo.psf -o run/demo_run -t 298 -e non -s 150
ls -la run


## 5. Visualize the trajectory

In [ ]:
!pip install -q py3Dmol MDAnalysis
import MDAnalysis as mda
import py3Dmol, glob

pdb_files = glob.glob("run/demo_run*.pdb") + ["run/demo.pdb"]
xtc_files = glob.glob("run/demo_run*.xtc")
print("PDB candidates:", pdb_files)
print("XTC candidates:", xtc_files)


In [ ]:
# Render the last frame of the trajectory (falls back to the built structure if no trajectory yet)
import glob

struct = glob.glob("run/demo_run*.pdb")
struct = struct[0] if struct else "run/demo.pdb"
traj = glob.glob("run/demo_run*.xtc")

if traj:
    u = mda.Universe(struct, traj[0])
    u.trajectory[-1]
    u.atoms.write("run/last_frame.pdb")
    view_file = "run/last_frame.pdb"
else:
    view_file = struct

with open(view_file) as f:
    pdb_block = f.read()

view = py3Dmol.view(width=600, height=500)
view.addModel(pdb_block, "pdb")
view.setStyle({"stick": {"radius": 0.3}, "sphere": {"scale": 0.3}})
view.zoomTo()
view.show()


## 6. Download your results

Zips up everything in `run/` (structure, PSF, trajectory, logs, checkpoint) so you can save it locally or attach it to your paper's supplementary data.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("hyres_results", "zip", "run")
files.download("hyres_results.zip")


---
## Notes for running this notebook

- Runtime disconnects after ~90 min idle on the free tier.
- Colab's free GPU (T4) is a shared, non-guaranteed resource — for production runs, follow [HyresBuilder repo](https://github.com/lslumass/HyresBuilder).
- Questions about the model itself: contact the ChenLab (jianhanc@umass.edu).
